# recurrentlens 01 — Quickstart

Load a Mamba-130M model, register a hook at the residual-stream analog (`out_proj_out`), and inspect captured activations.

This notebook runs on Colab CPU in ~2 minutes (download dominates). For SAE training, use notebook 03.

In [ ]:
%pip install -q recurrentlens[mamba] || %pip install -q git+https://github.com/hinanohart/recurrentlens.git

In [ ]:
import torch
import recurrentlens as rl
from recurrentlens.hooks.registry import HookManager

print('recurrentlens', rl.__version__)
print('hook sites:', rl.HOOK_SITES)

In [ ]:
# Load Mamba-130M (≈500 MB download on first run)
model = rl.load_model('state-spaces/mamba-130m-hf')
print(model)

In [ ]:
# Capture activations at layer 6, out_proj_out.
input_ids = model.tokenizer('The capital of France is', return_tensors='pt').input_ids.to(model.device)

with HookManager(model) as mgr:
    h = mgr.add(site='out_proj_out', layer=6, transform=lambda t: t.detach().cpu())
    with torch.no_grad():
        _ = model.forward(input_ids)

print('captured shape:', h.activations[0].shape)
print('mean:', float(h.activations[0].mean()), 'std:', float(h.activations[0].std()))

## Next steps

- **02_explore_pretrained.ipynb**: download a community SAE from the Hub and view top-activating features.
- **03_train_mamba130m_sae.ipynb**: train your own SAE on Colab T4 and push it to the Hub.